# Initialisation

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, lit, trim

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_custaz12")

# Silver Trnsformation

## Trimming
###### This code loops through all columns in a Spark DataFrame and trims whitespace from every string column to clean the data automatically.

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

## Customer ID Cleanup
###### This code removes the "NAS" prefix from cid values when present, otherwise leaves the value unchanged.

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
    .otherwise(col("cid")) 
    )

## Birthday Validation
###### This code validates dates by setting future dates to NULL while preserving valid historical dates.

In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
        .otherwise(col("bdate"))
)

## Gender Normalization

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
        .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
        .otherwise("n/a")
)

In [0]:
df.display()

## Renaming Columns

In [0]:
RENAME_MAP = {
    "cid" : "customer_id",
    "bdate" : "birth_date",
    "gen" : "gender"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# Write to Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

# Sanity Check

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10;

In [0]:
print(df.columns)